# 07 - Model: Weighted Mood Similarity

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "content_based"

tracks = pd.read_csv(PROCESSED_DIR / "content_tracks_engineered.csv")
print(tracks["mood_label"].value_counts())

mood_label
intense_dark        31292
bright_energetic    27410
calm_dark           23301
calm_positive        7737
Name: count, dtype: int64


## Feature space

In [2]:
MOOD_FEATURES = ["valence", "energy", "danceability", "acousticness", "instrumentalness", "tempo_log"]
MOOD_WEIGHTS = np.array([3.0, 3.0, 2.0, 1.0, 1.0, 0.5])

scaler = StandardScaler()
mood_matrix = scaler.fit_transform(tracks[MOOD_FEATURES]) * MOOD_WEIGHTS
print(mood_matrix.shape)

(89740, 6)


## Recommend

In [3]:
def recommend_by_mood(mood_label: str, top_n: int = 10, genre: str | None = None, similarity_weight: float = 0.5) -> pd.DataFrame:
    valid_moods = tracks["mood_label"].unique().tolist()
    if mood_label not in valid_moods:
        raise ValueError(f"Unknown mood_label {mood_label!r}. Choose from: {valid_moods}")

    mood_mask = tracks["mood_label"] == mood_label
    if genre:
        mood_mask &= tracks["track_genre"].str.contains(genre, case=False)

    candidate_indices = tracks.index[mood_mask].to_numpy()
    if len(candidate_indices) == 0:
        raise ValueError(f"No tracks found for mood={mood_label!r}, genre={genre!r}")

    centroid = mood_matrix[candidate_indices].mean(axis=0, keepdims=True)
    similarity = cosine_similarity(centroid, mood_matrix[candidate_indices])[0]
    popularity_norm = tracks.loc[candidate_indices, "popularity"].to_numpy() / 100

    combined_score = similarity_weight * similarity + (1 - similarity_weight) * popularity_norm
    ranked_order = combined_score.argsort()[::-1][:top_n]
    ranked_indices = candidate_indices[ranked_order]

    columns = ["track_id", "track_name", "artists", "track_genre", "popularity", "valence", "energy"]
    recommendations = tracks.loc[ranked_indices, columns].copy()
    recommendations["combined_score"] = combined_score[ranked_order]
    return recommendations.reset_index(drop=True)

In [4]:
for mood in tracks["mood_label"].unique():
    print(f"\n{mood}")
    display(recommend_by_mood(mood, top_n=5))


calm_positive


,track_id,track_name,artists,track_genre,popularity,valence,energy,combined_score
0,2374M0fQpWi3dLnB54qaLX,Africa,TOTO,rock,83,0.732,0.373,0.899709
1,0snQrp1VaY5Pj1YIHRJpRJ,Chaand Baaliyan,Aditya A,indie,81,0.886,0.396,0.891950
2,1EzrEOXmMH3G43AXT1y7pA,I'm Yours,Jason Mraz,acoustic,80,0.712,0.444,0.887801
3,1JSTJqkT5qHq8MDJnJbRE1,Every Breath You Take,The Police,rock,86,0.740,0.452,0.886003
4,3fjN3y5x4hN53rykAN2LHQ,cómo dormiste?,Rels B,latino,89,0.565,0.429,0.885838



calm_dark


,track_id,track_name,artists,track_genre,popularity,valence,energy,combined_score
0,6xGruZOHLs39ZbVccQTuPZ,Glimpse of Us,Joji,pop,94,0.268,0.317,0.948803
1,3WMj8moIAXJhHsyLaqIIHI,Something in the Orange,Zach Bryan,country,89,0.148,0.192,0.928187
2,0u2P5u6lvoDfwTYjAADbn4,lovely (with Khalid),Billie Eilish;Khalid,pop,89,0.120,0.296,0.917398
3,4RVwu0g32PAqgUiJoXsdF8,Happier Than Ever,Billie Eilish,pop,88,0.297,0.225,0.910772
4,5Qnrgqy1pAm9GyNQOgyVFz,Fourth of July,Sufjan Stevens,singer-songwriter,81,0.162,0.104,0.903384



intense_dark


,track_id,track_name,artists,track_genre,popularity,valence,energy,combined_score
0,4uUG5RXrOk84mYEfFvj3cK,I'm Good (Blue),David Guetta;Bebe Rexha,dance,98,0.304,0.965,0.959807
1,0BxE4FqsDD1Ot4YuBXwAPp,505,Arctic Monkeys,garage,88,0.248,0.866,0.932179
2,57bgtoPSgt236HzfBOd8kj,Thunderstruck,AC/DC,hard-rock,84,0.259,0.890,0.912766
3,62ke5zFUJN6RvtXZgVH0F8,Only Love Can Hurt Like This,Paloma Faith,british,87,0.304,0.885,0.909417
4,2nLtzopw4rPReszdYBJU6h,Numb,Linkin Park,alternative,83,0.243,0.863,0.906756



bright_energetic


,track_id,track_name,artists,track_genre,popularity,valence,energy,combined_score
0,4h9wh7iOZ0GGn8QVp4RAOB,I Ain't Worried,OneRepublic,piano,96,0.825,0.797,0.975456
1,0WtM2NBVQNNJLh6scP13H8,Calm Down (with Selena Gomez),Rema;Selena Gomez,pop,92,0.802,0.806,0.931989
2,0xzI1KAr0Yd9tv8jlIk3sn,Bad Decisions (with BTS & Snoop Dogg),benny blanco;BTS;Snoop Dogg,dance,87,0.955,0.861,0.931116
3,2gYj9lubBorOPIVWsTXugG,After LIKE,IVE,k-pop,88,0.799,0.922,0.926888
4,4C6Uex2ILwJi9sZXRdmqXp,Super Freaky Girl,Nicki Minaj,dance,92,0.912,0.891,0.922168


## Save

In [5]:
np.save(MODEL_DIR / "weighted_feature_matrix.npy", mood_matrix)
joblib.dump(scaler, MODEL_DIR / "weighted_scaler.joblib")

weighted_config = {
    "model": "weighted_mood_similarity",
    "use_case": "mood",
    "features": MOOD_FEATURES,
    "weights": MOOD_WEIGHTS.tolist(),
    "similarity_weight": 0.5,
    "candidate_pool": "same_mood_label",
}
with open(MODEL_DIR / "weighted_config.json", "w", encoding="utf-8") as file:
    json.dump(weighted_config, file, indent=2)